In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from stl import Mesh
from base import iterative_section_minimiser

In [ ]:
# read the mesh file and its adjacent triangles. 
mesh_data = Mesh.from_file("data/cylinder_mesh/straight_cylinder_L10_R1.stl")

In [ ]:
def compare_analytical(n):
    
    n_norm = n /  np.linalg.norm(n)
    
    a = iterative_section_minimiser.section_area(1, n_norm)
    W = iterative_section_minimiser.analytical_W(1, n_norm)

    _,_,_, W2, a2, _ = iterative_section_minimiser.mesh_intersect(mesh_data, [0,0,0], n_norm)

    return a, a2, W, W2


In [ ]:
# test the minimiser against the analytical solution for a cylinder of radius 1, in the x direction. 
test_normals = np.zeros((10, 3))

test_normals[:, 0] = np.ones(10)
test_normals[:, 1] = np.linspace(0.1, 3.0, 10)

thetai = [elem[0] / np.linalg.norm(elem) for elem in test_normals]
test_outputs = [compare_analytical(n) for n in test_normals]


In [ ]:
fig, axes = plt.subplots(2, 1)

axes[0].scatter(thetai, [elem[0] for elem in  test_outputs], label = "analytical solution")
axes[0].scatter(thetai, [elem[1] for elem in  test_outputs], label = "numerical estimation from mesh", marker = "+")
axes[0].set_ylabel("area")
axes[0].set_xlabel("$cos\\theta$")

axes[1].scatter(thetai, [np.linalg.norm(elem[2]) for elem in  test_outputs], label = "analytical")
axes[1].scatter(thetai, [np.linalg.norm(elem[3]) for elem in  test_outputs], label = "numerical estimation from mesh", marker = "+")
axes[1].set_ylabel("||W||")
axes[1].set_xlabel("$cos\\theta$")

for ax in axes: 
    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
# read the mesh file and its adjacent triangles. 
#conduit_type = "sinuous_D5_r_1"
conduit_type = "straight_cylinder_L8_R1_stochastic"
mesh_data = Mesh.from_file(f"data/cylinder_mesh/{conduit_type}.stl")

P0 = np.array([-3.9, 0., 0])
P_end = np.array([3.9, -0., 0])
nP0 = np.array([1, 0, 0])
L = 10
ds = 0.1
verbose = True


In [ ]:
result = iterative_section_minimiser.iterative_pathfinder(mesh_data, P0, P_end, nP0, verbose = False, ds=ds)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize = (6, 12))

for viewpoint, ax in zip([[0,1],[0,2], [1,2]], axes):
    for c, step in enumerate(result["vertices"]):
        ax.plot(*step[:, viewpoint].T, color = "deeppink", lw = 0.5)

        # plot the point cloud
        ax.scatter(*mesh_data[::50, viewpoint].T,
                   color = "lightgrey",
                   alpha = 0.5, 
                   s=2)

        # plot the centreline
        ax.plot(*result["centreline"][:, viewpoint].T, 
                "-o", 
                markersize =3, 
                color = "deeppink", 
                zorder = 10)
        
        
        # plot the centreline
        for c, elem in enumerate(result["centreline"][:-1]):

            ax.text(*elem[viewpoint],
                    result["n_iterations"][c], 
                    fontsize =7, 
                    color = "deeppink", 
                    zorder = 10)
        
        # plot the vector normals
        
        ax.quiver(*result["centreline"][:-1, viewpoint].T, 
                  *result["normals"][:, viewpoint].T, 
                  color ="dodgerblue",
                  width = 0.002, 
                  zorder = 100)

        ax.set_aspect("equal")

plt.tight_layout()
plt.savefig(f"temp/IterativeCentreline_{conduit_type}_plot.png", dpi = 300)